In [1]:
# Imports and path config for merged dataset preprocessing.
from pathlib import Path
import pandas as pd
import re

DATA_DIR = Path("/Users/jamesemcnally/Documents/GitHub/critical-listener/datasets")

df = pd.read_csv(DATA_DIR / "merged_dataset.csv", low_memory=False)
print(f"Shape: {df.shape}")
print(f"\nDataset counts:")
print(df["dataset"].value_counts())


Shape: (48639, 22)

Dataset counts:
dataset
pitchfork           22853
resident_advisor    15560
critique_brainz     10226
Name: count, dtype: int64


### Clean merged dataset before deep learning

In [3]:
# Broad audit of CritiqueBrainz text quality before cleaning.
# Surfaces boilerplate, HTML remnants, URLs, and repeated sentence fragments
# across all CB reviews — not just BBC-sourced ones.

import re
from collections import Counter

cb_text = df[df["dataset"] == "critique_brainz"]["cleaned_text"].dropna()

print("=== 1. URL / link artifacts ===")
url_pattern = r'http\S+|www\.\S+'
url_hits = cb_text[cb_text.str.contains(url_pattern, regex=True, na=False)]
print(f"Reviews containing URLs: {len(url_hits):,}")
for t in url_hits.head(3):
    urls = re.findall(url_pattern, t)
    print(f"  {urls}")

print("\n=== 2. HTML remnants ===")
html_pattern = r'<[^>]+>|&[a-z]+;|&#\d+;'
html_hits = cb_text[cb_text.str.contains(html_pattern, regex=True, na=False)]
print(f"Reviews with HTML tags or entities: {len(html_hits):,}")
for t in html_hits.head(3):
    found = re.findall(html_pattern, t)
    print(f"  {found[:5]}")

print("\n=== 3. Star/numeric rating strings ===")
rating_pattern = r'\b[★☆✩✭]|\b\d[\./]\d\s*(out of|\/)\s*\d|\brating\s*:\s*\d'
rating_hits = cb_text[cb_text.str.contains(rating_pattern, regex=True, na=False)]
print(f"Reviews with rating strings: {len(rating_hits):,}")
for t in rating_hits.head(3):
    print(f"  {repr(t[:200])}")

print("\n=== 4. Repeated sentences across reviews (likely boilerplate) ===")
# Split each review into sentences, collect all, find those that appear in many reviews
all_sentences = []
for text in cb_text:
    sentences = [s.strip() for s in re.split(r'(?<=[.!?])\s+', text) if len(s.strip()) > 20]
    all_sentences.extend(sentences)

sentence_counts = Counter(all_sentences)
print(f"Top 20 most-repeated sentences across CB reviews:")
for sentence, count in sentence_counts.most_common(20):
    print(f"  [{count:>4}x] {repr(sentence[:120])}")

print("\n=== 5. First and last 150 chars of 10 random CB reviews ===")
for i, text in enumerate(cb_text.sample(10, random_state=7)):
    print(f"\n--- Review {i+1} ---")
    print(f"  START: {repr(text[:150])}")
    print(f"  END:   {repr(text[-150:])}")


=== 1. URL / link artifacts ===
Reviews containing URLs: 52
  ['www.MainlyPiano.com']
  ['www.MainlyPiano.com']
  ['www.MainlyPiano.com']

=== 2. HTML remnants ===
Reviews with HTML tags or entities: 7
  ['&nbsp;', '&nbsp;']
  ['&b;']
  ['&b;']

=== 3. Star/numeric rating strings ===


/var/folders/1v/h2vyfq9j45n9_tmm_gg2s8tm0000gn/T/ipykernel_60691/437123359.py:28: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  rating_hits = cb_text[cb_text.str.contains(rating_pattern, regex=True, na=False)]


Reviews with rating strings: 6
  'BWET is THE best alt-rap album of the 2010s, whilst also being able to incorperate elements of Salsa (Gimmie A Chance), and UK Trap (BBD). Azealia is a very versitile artist, her personality and out-s'
  'Album Review Phenomenal album for neuro-lovers. I think this is one of the most deep and emotional piece made by the duo, with incredibly powerful tracks coupled with insane sound design as expected b'
  'Broods\' latest installment in pop scene comes with viral, addcitive, and emotionally-stricken songs. Aside from the sure-bop "Peach" other stand-out songs comes from the personal "Too Proud" and the l'

=== 4. Repeated sentences across reviews (likely boilerplate) ===
Top 20 most-repeated sentences across CB reviews:
  [ 168x] "\\- - - Follow the BBC's Album Reviews service on Twitter"
  [  26x] 'Andrew McGregor - presenter of CD Review on Radio 3'
  [  12x] "\\- - - Follow the BBC's album reviews service on Twitter"
  [  10x] "This recording is Dis

In [4]:
# Strip CritiqueBrainz boilerplate and HTML artifacts.
# Applies patterns in order: most specific first to avoid over-stripping.
# Only modifies cleaned_text for CB rows; other datasets untouched.

STRIP_PATTERNS = [
    # BBC footer / attribution lines (the \- - - prefix variants)
    r'\\?-\s*-\s*-\s*Follow the BBC.{0,60}Twitter\.?',
    r'\\?-\s*-\s*-\s*This album is reviewed on.{0,80}',
    r'\\?-\s*-\s*-\s*[A-Z].{0,120}',       # catch any remaining \- - - prefixed lines
    r'Andrew McGregor\s*-\s*presenter of CD Review on Radio 3[^\n]*',
    r'This recording is Disc Of The Week on[^\n]*',
    r'Review courtesy of[^\n]*',
    # BBC "Like This? Try These:" widget — strip from marker to end of that block
    r'Like This\?\s*Try These\s*:.*',
    # Metal Archives attribution
    r'originally written for (metallum|metal archives)[^\n]*',
    # Social media hashtag strings (3+ consecutive hashtags)
    r'(#\w+\s*){3,}',
    # Reviewer attribution lines
    r'Reviwer?r?\s*:\s*[A-Z][^\n]*',
    # MainlyPiano.com URLs and surrounding text
    r'www\.MainlyPiano\.com[^\s]*',
    # HTML entities
    r'&nbsp;|&b;|&[a-z]+;|&#\d+;',
]

def strip_boilerplate(text):
    for pattern in STRIP_PATTERNS:
        text = re.sub(pattern, ' ', text, flags=re.IGNORECASE)
    # Collapse multiple spaces/newlines left behind by removals
    text = re.sub(r'\s{2,}', ' ', text).strip()
    return text

cb_mask = df["dataset"] == "critique_brainz"

# Apply only to CB rows
original_cb = df.loc[cb_mask, "cleaned_text"].copy()
df.loc[cb_mask, "cleaned_text"] = df.loc[cb_mask, "cleaned_text"].dropna().apply(strip_boilerplate)

# Recompute char_count for CB rows
df.loc[cb_mask, "char_count"] = df.loc[cb_mask, "cleaned_text"].str.len()

# --- Stats ---
chars_before = original_cb.str.len()
chars_after  = df.loc[cb_mask, "cleaned_text"].str.len()
affected     = (chars_before != chars_after).sum()
too_short    = (df.loc[cb_mask, "char_count"] < 200).sum()

print(f"CB reviews modified:                {affected:,}")
print(f"CB reviews now under 200 chars:     {too_short:,}  (will be dropped next cell)")
print(f"Mean chars removed per review:      {(chars_before - chars_after).mean():.1f}")

# --- Before/after spot check ---
changed_idx = original_cb.index[(chars_before != chars_after)]
print(f"\n--- Before/After samples (3 most-changed reviews) ---")
top3 = (chars_before - chars_after).nlargest(3).index
for idx in top3:
    print(f"\nBEFORE: ...{repr(original_cb[idx][-300:])}")
    print(f"AFTER:  ...{repr(df.loc[idx, 'cleaned_text'][-300:])}")


CB reviews modified:                3,147
CB reviews now under 200 chars:     1  (will be dropped next cell)
Mean chars removed per review:      5.7

--- Before/After samples (3 most-changed reviews) ---

BEFORE: ...'e contained in a thin cardboard slipcase. This is the most comprehensive collection of Ultraman Ace music available now. That said, this set would probably appeal more to completists and hardcore fans. Casual fans will be more than satisfied with one of the many single disk releases still available.'
AFTER:  ...'April 12, 2019:'

BEFORE: ...' tracks not included in the game at all, but studio recordings (taken from the official soundtrack?), that nevertheless fit in well with the rest, and is probably worth checking out in its own right. Overall a very distinct effort, that, for better or worse, confidently stands apart from the others.'
AFTER:  ...' tracks not included in the game at all, but studio recordings (taken from the official soundtrack?), that nevertheless fit i

In [5]:
# Drop CB reviews that fell below 200 characters after boilerplate removal.
# Also recompute word_count to stay consistent with char_count update above.

cb_mask = df["dataset"] == "critique_brainz"

before = cb_mask.sum()
df = df[~(cb_mask & (df["char_count"] < 200))].reset_index(drop=True)
after = (df["dataset"] == "critique_brainz").sum()

# Recompute word_count for CB rows now that cleaned_text has changed
cb_mask = df["dataset"] == "critique_brainz"
df.loc[cb_mask, "word_count"] = df.loc[cb_mask, "cleaned_text"].str.split().str.len()

print(f"CB reviews before drop: {before:,}")
print(f"CB reviews after drop:  {after:,}  ({before - after} removed)")
print(f"\nDataset counts after cleaning:")
print(df["dataset"].value_counts())
print(f"\nTotal reviews: {len(df):,}")


CB reviews before drop: 10,226
CB reviews after drop:  10,225  (1 removed)

Dataset counts after cleaning:
dataset
pitchfork           22853
resident_advisor    15560
critique_brainz     10225
Name: count, dtype: int64

Total reviews: 48,638


In [6]:
# Deduplicate: keep the longest review per artist/album/source combination.
# Creates artist_norm and album_norm (lowercased, stripped) for matching.
# These columns are retained for use in the "Various" normalization step.

df["artist_norm"] = df["artist"].str.lower().str.strip()
df["album_norm"]  = df["album"].str.lower().str.strip()

before = len(df)

# Rank by char_count descending within each (dataset, artist, album) group;
# keep only rank 1 (the longest review)
df["_rank"] = (
    df.groupby(["dataset", "artist_norm", "album_norm"])["char_count"]
    .rank(method="first", ascending=False)
)
df = df[df["_rank"] == 1].drop(columns="_rank").reset_index(drop=True)

after = len(df)

print(f"Reviews before dedup: {before:,}")
print(f"Reviews after dedup:  {after:,}  ({before - after} removed)")
print(f"\nBy dataset:")
print(df["dataset"].value_counts())


Reviews before dedup: 48,638
Reviews after dedup:  48,189  (449 removed)

By dataset:
dataset
pitchfork           22810
resident_advisor    15318
critique_brainz     10061
Name: count, dtype: int64


In [7]:
# Diagnostic: verify null rows and duplicate count before accepting the dedup result.

has_key = df["artist_norm"].notna() & df["album_norm"].notna()

print(f"Null artist_norm rows: {df['artist_norm'].isna().sum():,}")
print(f"Null album_norm rows:  {df['album_norm'].isna().sum():,}")
print(f"Rows excluded from dedup (df_null): {(~has_key).sum():,}")
print(f"Rows included in dedup (df_keyed):  {has_key.sum():,}")
print()

dupes = (
    df[has_key]
    .groupby(["dataset", "artist_norm", "album_norm"])
    .size()
    .reset_index(name="count")
)
dupes = dupes[dupes["count"] > 1]
print(f"Duplicate pairs found: {len(dupes):,}")
print(f"Excess reviews:        {(dupes['count'] - 1).sum():,}")
print()
print("By dataset:")
print(dupes.groupby("dataset").apply(lambda x: (x["count"] - 1).sum(), include_groups=False))


Null artist_norm rows: 0
Null album_norm rows:  0
Rows excluded from dedup (df_null): 0
Rows included in dedup (df_keyed):  48,189

Duplicate pairs found: 0
Excess reviews:        0

By dataset:
Empty DataFrame
Columns: [artist_norm, album_norm, count]
Index: []


In [8]:
# Inspect artist values that represent compilations or non-standard entries.
# "Various Artists" etc. are now strings, not nulls — find and document them.

low_count_artists = df["artist_norm"].value_counts()

print("Potential compilation / non-artist values (low count or known patterns):")
print(low_count_artists[
    low_count_artists.index.str.contains(
        r'^various|^va$|^multiple|^unknown|^compilation',
        case=False, na=False, regex=True
    )
].to_string())

print(f"\nTotal reviews with 'various'-type artist: "
      f"{low_count_artists[low_count_artists.index.str.contains(r'^various|^va$|^multiple|^unknown|^compilation', case=False, na=False, regex=True)].sum():,}")


Potential compilation / non-artist values (low count or known patterns):
artist_norm
various artists             1551
various                      395
unknown artist                21
unknown                       11
unknown mortal orchestra       7
various production             4
va                             4
unknown mobile                 2
unknown archetype              1
various cruelties              1

Total reviews with 'various'-type artist: 1,997


In [9]:
# Normalize genre tag capitalization to title case.
# Fixes inconsistencies like "experimental" vs "Experimental" vs "EXPERIMENTAL".
# RA reviews keep their sub-genre tags (e.g., "Techno", "Deep House") — not collapsed here.

before = df["genre"].dropna().unique()

df["genre"] = df["genre"].str.title()

after = df["genre"].dropna().unique()

print(f"Unique genre values before: {len(before):,}")
print(f"Unique genre values after:  {len(after):,}")
print(f"\nSample genre values after normalization:")
print(sorted(df["genre"].dropna().unique())[:30])


Unique genre values before: 1,135
Unique genre values after:  1,132

Sample genre values after normalization:
['Acid', 'Acid, Breakcore', 'Afrobeat', 'Ambient', 'Ambient, Acid', 'Ambient, Balearic', 'Ambient, Classical', 'Ambient, Classical, Drone', 'Ambient, Deephouse', 'Ambient, Deephouse, Downtempo', 'Ambient, Deephouse, Experimental', 'Ambient, Disco', 'Ambient, Disco, Balearic', 'Ambient, Disco, Downtempo', 'Ambient, Disco, Experimental, Pop, Italodisco', 'Ambient, Disco, Funksoul', 'Ambient, Downtempo', 'Ambient, Downtempo, Dubstep', 'Ambient, Downtempo, Experimental', 'Ambient, Downtempo, Experimental, Idm', 'Ambient, Drone', 'Ambient, Drone, Noise', 'Ambient, Dub', 'Ambient, Dub, Experimental', 'Ambient, Dubstep', 'Ambient, Dubstep, Dancehall', 'Ambient, Dubstep, Experimental', 'Ambient, Experimental', 'Ambient, Experimental, Classical', 'Ambient, Experimental, Club']


### Save cleaned dataset to CSV

In [10]:
# Save cleaned dataset to a new CSV — keeps the original merged_dataset.csv untouched.

out_path = DATA_DIR / "merged_dataset_cleaned.csv"
df.to_csv(out_path, index=False)

print(f"Saved: {out_path}")
print(f"Shape: {df.shape}")
print(f"\nFinal dataset counts:")
print(df["dataset"].value_counts())
print(f"\nColumns: {list(df.columns)}")


Saved: /Users/jamesemcnally/Documents/GitHub/critical-listener/datasets/merged_dataset_cleaned.csv
Shape: (48189, 24)

Final dataset counts:
dataset
pitchfork           22810
resident_advisor    15318
critique_brainz     10061
Name: count, dtype: int64

Columns: ['dataset', 'review_id', 'entity_id', 'text', 'rating', 'album', 'artist', 'reviewer_name', 'reviewer_id', 'reviewer_type', 'reviewer_karma', 'published_on', 'source', 'source_url', 'cleaned_text', 'word_count', 'char_count', 'sentence_count', 'sentiment_score', 'sentiment_category', 'genre', 'ra_recommends', 'artist_norm', 'album_norm']


### Model Exploration

In [11]:
# Quick-start cell: load the cleaned dataset directly without re-running preprocessing.
# Uncomment and run this if starting a fresh session in this notebook.

# from pathlib import Path
# import pandas as pd
# 
# DATA_DIR = Path("/Users/jamesemcnally/Documents/GitHub/critical-listener/datasets")
# df = pd.read_csv(DATA_DIR / "merged_dataset_cleaned.csv", low_memory=False)
# print(f"Loaded: {df.shape[0]:,} reviews")
# print(df["dataset"].value_counts())

In [12]:
# Create two versions of model input: one with artist/album prefix, one without.
# The two versions let us measure how much each model relies on the album name vs. the review content itself.
df["input_with_prefix"] = (
    "Artist: " + df["artist"].fillna("Unknown") + ". " +
    "Album: "  + df["album"].fillna("Unknown")  + ". " +
    df["cleaned_text"]
)
df["input_no_prefix"] = df["cleaned_text"]

print(df["input_with_prefix"].iloc[10][:200])


Artist: AC/DC. Album: Back in Black. Back in Black is one of those albums where you instantly understand why it became legendary. The riffs are huge, the rhythm section is locked in, and the whole thi


In [13]:
# Find every album reviewed by more than one source and pair those reviews together.
# These pairs are used only to select the best model — not to prove recommendation quality.
# Limitation: a Pitchfork critic and an amateur reviewer may describe the same album very differently,
# so a model that excels at deep semantic similarity may actually score lower here than one
# that relies on surface vocabulary overlap. This is a known tradeoff we accept for model selection.
import numpy as np
from itertools import combinations

eval_pairs = []
keyed = df.dropna(subset=["artist_norm", "album_norm"])
for _, group in keyed.groupby(["artist_norm", "album_norm"]):
    if group["dataset"].nunique() > 1:
        for i, j in combinations(group.index.tolist(), 2):
            if df.loc[i, "dataset"] != df.loc[j, "dataset"]:
                eval_pairs.append((i, j))

print(f"Evaluation pairs: {len(eval_pairs):,}")


Evaluation pairs: 3,653


In [17]:
# Measure how highly each model ranks the correct match for a given query review.
# We use two metrics: Recall@k (did the right answer appear in the top k results?)
# and MRR (how high did it rank on average?). Higher is better for both.
def evaluate_retrieval(get_sims_fn, eval_pairs, ks=(1, 5, 10)):
    rr, hits = [], {k: [] for k in ks}
    for q_idx, t_idx in eval_pairs:
        sims = get_sims_fn(q_idx).copy()
        sims[q_idx] = -np.inf        # don't retrieve the query itself

        # save the target's score before masking, then restore it after
        # (target is same album but different source — we want to find it)
        target_sim = sims[t_idx]
        same_album = df[
            (df["artist_norm"] == df.loc[q_idx, "artist_norm"]) &
            (df["album_norm"]  == df.loc[q_idx, "album_norm"])
        ].index
        sims[same_album] = -np.inf
        sims[t_idx] = target_sim

        ranked = np.argsort(sims)[::-1]
        rank = int(np.where(ranked == t_idx)[0][0]) + 1
        rr.append(1.0 / rank)
        for k in ks:
            hits[k].append(int(rank <= k))
    return {"MRR": np.mean(rr), **{f"Recall@{k}": np.mean(hits[k]) for k in ks}}


In [18]:
# Fit TF-IDF on all reviews and compute cosine similarity between reviews.
# TF-IDF is our floor — the simplest possible approach, no neural network.
# Any embedding model we try should beat this; if it doesn't, something is wrong.
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity as cos_sim

results = {}

for input_col, label in [("input_with_prefix", "TF-IDF (with prefix)"),
                          ("input_no_prefix",   "TF-IDF (no prefix)")]:
    vectorizer = TfidfVectorizer(max_features=50_000, min_df=2, stop_words="english")
    matrix = vectorizer.fit_transform(df[input_col])

    def tfidf_sims(idx, matrix=matrix):
        return cos_sim(matrix[idx], matrix).flatten()

    results[label] = evaluate_retrieval(tfidf_sims, eval_pairs)
    print(f"\n{label}")
    print("-" * 35)
    for metric, val in results[label].items():
        print(f"  {metric}: {val:.4f}")



TF-IDF (with prefix)
-----------------------------------
  MRR: 0.7942
  Recall@1: 0.6868
  Recall@5: 0.9381
  Recall@10: 0.9828

TF-IDF (no prefix)
-----------------------------------
  MRR: 0.7164
  Recall@1: 0.5905
  Recall@5: 0.8875
  Recall@10: 0.9518


In [25]:
# Save as parquet — handles embedded newlines and quotes cleanly.
df.to_parquet(DATA_DIR / "merged_dataset_cleaned.parquet", index=False)


In [26]:
import os
print(os.path.getsize(DATA_DIR / "merged_dataset_cleaned.parquet"))


386716552


In [28]:
!pip install psycopg2-binary


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 11.2 MB/s  0:00:00eta 0:00:01


In [38]:
# Check how many albums in the dataset can be matched to MusicBrainz releases.
# Tests exact match on lowercased album name and artist name.
import psycopg2
import pandas as pd

conn = psycopg2.connect(
    dbname="musicbrainz_db",
    user="musicbrainz",
    password="musicbrainz",
    host="localhost",
    port=5432
)

# Get a sample of 20 albums from the dataset to test matching
sample = df[df["dataset"] == "pitchfork"].dropna(subset=["artist_norm", "album_norm"]).sample(20, random_state=42)[["artist", "album", "artist_norm", "album_norm"]]

results = []
for _, row in sample.iterrows():
    try:
        cur = conn.cursor()
        cur.execute("""
            SET statement_timeout = '5s';
            SELECT r.id, r.name, a.name as artist_name
            FROM release r
            JOIN artist_credit ac ON r.artist_credit = ac.id
            JOIN artist_credit_name acn ON ac.id = acn.artist_credit
            JOIN artist a ON acn.artist = a.id
            WHERE lower(r.name) = %s
            AND lower(a.name) = %s
            LIMIT 1
        """, (row["album_norm"], row["artist_norm"]))
        match = cur.fetchone()
        results.append({
            "album":    row["album"],
            "artist":   row["artist"],
            "matched":  match is not None,
            "mb_album": match[1] if match else None,
            "mb_artist": match[2] if match else None
        })
    except Exception as e:
        results.append({"album": row["album"], "artist": row["artist"], "matched": False, "mb_album": None, "mb_artist": str(e)})

conn.close()

results_df = pd.DataFrame(results)
print(f"Match rate: {results_df['matched'].mean()*100:.1f}%")
print()
print(results_df.to_string(index=False))


Match rate: 75.0%

                                         album                        artist  matched               mb_album       mb_artist
                                          1983                        Kölsch     True                   1983          Kölsch
                                          LUMP                          LUMP     True                   LUMP            LUMP
Thelonious Monk: Les Liaisons Dangereuses 1960               Thelonious Monk    False                   None            None
                                      The Trip  Jarvis Cocker & Steve Mackey    False                   None            None
                          The Boy and the Tree                 Susumu Yokota     True   The Boy and the Tree   Susumu Yokota
          New Amerykah Part One: 4th World War                   Erykah Badu    False                   None            None
                      Trade Test Tramsmissions                     Buzzcocks    False                   No

In [39]:
# Test personnel query on a single album before scaling to full corpus.
# Gets: credited artist, band members, and producer for a given release.
import psycopg2

conn = psycopg2.connect(
    dbname="musicbrainz_db",
    user="musicbrainz",
    password="musicbrainz",
    host="localhost",
    port=5432
)

def get_personnel(album_norm, artist_norm, conn):
    cur = conn.cursor()

    # Step 1: find the release and credited artist
    cur.execute("""
        SET statement_timeout = '10s';
        SELECT r.id, a.id, a.name
        FROM release r
        JOIN artist_credit ac ON r.artist_credit = ac.id
        JOIN artist_credit_name acn ON ac.id = acn.artist_credit
        JOIN artist a ON acn.artist = a.id
        WHERE lower(r.name) = %s AND lower(a.name) = %s
        LIMIT 1
    """, (album_norm, artist_norm))
    row = cur.fetchone()
    if not row:
        return None
    release_id, artist_id, artist_name = row
    names = {artist_name}

    # Step 2: get band members of the credited artist
    cur.execute("""
        SELECT a.name
        FROM l_artist_artist laa
        JOIN link l ON laa.link = l.id
        JOIN link_type lt ON l.link_type = lt.id
        JOIN artist a ON laa.entity0 = a.id
        WHERE laa.entity1 = %s AND lt.name = 'member of band'
    """, (artist_id,))
    for (name,) in cur.fetchall():
        names.add(name)

    # Step 3: get producer linked directly to the release
    cur.execute("""
        SELECT a.name
        FROM l_artist_release lar
        JOIN link l ON lar.link = l.id
        JOIN link_type lt ON l.link_type = lt.id
        JOIN artist a ON lar.entity0 = a.id
        WHERE lar.entity1 = %s AND lt.name IN ('producer', 'co-producer')
    """, (release_id,))
    for (name,) in cur.fetchall():
        names.add(name)

    return sorted(names)

# Test on Kid A
result = get_personnel("kid a", "radiohead", conn)
print(f"Kid A personnel: {result}")

conn.close()


Kid A personnel: ['Colin Greenwood', 'Ed O’Brien', 'Jonny Greenwood', 'Philip Selway', 'Radiohead', 'Thom Yorke']


In [42]:
# Create index on lowercased release name to speed up the bulk query.
# One-time cost — takes 5-10 minutes but makes the query fast afterwards.
conn = psycopg2.connect(
    dbname="musicbrainz_db",
    user="musicbrainz",
    password="musicbrainz",
    host="localhost",
    port=5432
)
cur = conn.cursor()
cur.execute("CREATE INDEX IF NOT EXISTS idx_release_name_lower ON release (lower(name));")
conn.commit()
conn.close()
print("Index created.")


Index created.


In [43]:
# Bulk personnel query: uploads all target pairs to a temp table,
# runs one JOIN against MusicBrainz, returns all personnel at once.
from collections import defaultdict
import psycopg2
import psycopg2.extras

conn = psycopg2.connect(
    dbname="musicbrainz_db",
    user="musicbrainz",
    password="musicbrainz",
    host="localhost",
    port=5432
)
cur = conn.cursor()

# Step 1: create temp table and bulk insert target pairs
cur.execute("""
    CREATE TEMP TABLE target_albums (
        artist_norm TEXT,
        album_norm  TEXT
    )
""")

unique_pairs = df.dropna(subset=["artist_norm", "album_norm"])[
    ["artist_norm", "album_norm"]
].drop_duplicates().values.tolist()

psycopg2.extras.execute_values(
    cur,
    "INSERT INTO target_albums (artist_norm, album_norm) VALUES %s",
    unique_pairs
)
conn.commit()
print(f"Inserted {len(unique_pairs):,} pairs into temp table")

# Step 2: bulk query — credited artist + band members in one shot
cur.execute("""
    SET statement_timeout = '300s';

    SELECT
        t.artist_norm,
        t.album_norm,
        a.name       AS personnel_name,
        'credited'   AS role
    FROM target_albums t
    JOIN release r              ON lower(r.name)    = t.album_norm
    JOIN artist_credit ac       ON r.artist_credit  = ac.id
    JOIN artist_credit_name acn ON ac.id            = acn.artist_credit
    JOIN artist a               ON acn.artist       = a.id
    WHERE lower(a.name) = t.artist_norm

    UNION ALL

    SELECT
        t.artist_norm,
        t.album_norm,
        a_member.name AS personnel_name,
        'member'      AS role
    FROM target_albums t
    JOIN release r              ON lower(r.name)      = t.album_norm
    JOIN artist_credit ac       ON r.artist_credit    = ac.id
    JOIN artist_credit_name acn ON ac.id              = acn.artist_credit
    JOIN artist a_group         ON acn.artist         = a_group.id
    JOIN l_artist_artist laa    ON laa.entity1        = a_group.id
    JOIN link l                 ON laa.link           = l.id
    JOIN link_type lt           ON l.link_type        = lt.id
    JOIN artist a_member        ON laa.entity0        = a_member.id
    WHERE lower(a_group.name) = t.artist_norm
    AND lt.name = 'member of band'
""")

rows = cur.fetchall()
conn.close()
print(f"Retrieved {len(rows):,} personnel records")

# Step 3: aggregate into a dict keyed by (artist_norm, album_norm)
personnel_map = defaultdict(set)
for artist_norm, album_norm, name, role in rows:
    personnel_map[(artist_norm, album_norm)].add(name)
personnel_map = {k: sorted(v) for k, v in personnel_map.items()}

# Step 4: add personnel column to df
df["personnel"] = df.apply(
    lambda row: personnel_map.get((row["artist_norm"], row["album_norm"])),
    axis=1
)

matched = df["personnel"].notna().sum()
print(f"Reviews with personnel data: {matched:,} ({matched/len(df)*100:.1f}%)")
print("\nSample:")
print(df[df["personnel"].notna()][["artist", "album", "personnel"]].head(5).to_string(index=False))


Inserted 44,716 pairs into temp table
Retrieved 591,648 personnel records
Reviews with personnel data: 32,757 (68.0%)

Sample:
             artist                   album                                                                                                                                                                                                                                                                                                                                                 personnel
       Astrid Hadad    La Pluma o La Espada                                                                                                                                                                                                                                                                                                                                            [Astrid Hadad]
      Guns N’ Roses     Use Your Illusion I [Axl Rose, Bryan “Brain” Mantia, Buckethead, Chri

In [44]:
# Create masked review text by replacing personnel names with [MASKED].
# Preserves original cleaned_text — only adds a new cleaned_text_masked column.
import re

def mask_personnel(text, personnel):
    if not personnel or not isinstance(text, str):
        return text
    for name in personnel:
        if not name or len(name) < 3:
            continue
        pattern = re.compile(re.escape(name), re.IGNORECASE)
        text = pattern.sub("[MASKED]", text)
    return text

df["cleaned_text_masked"] = df.apply(
    lambda row: mask_personnel(row["cleaned_text"], row["personnel"]),
    axis=1
)

# Spot check on Radiohead
radiohead = df[(df["artist_norm"] == "radiohead") & (df["album_norm"] == "kid a")].iloc[0]
print("Original:")
print(radiohead["cleaned_text"][:500])
print("\nMasked:")
print(radiohead["cleaned_text_masked"][:500])


Original:
By 1998 Radiohead were at breaking point. Mentally and physically exhausted, Thom Yorke – forever the sickly Victorian child – had been pushed too far by fame and adulation. Like many before him, the thrill of playing to a stadium full of unquestioningly adoring fans had become soured by the sheer numbing quality of constant public scrutiny. In a way, Radiohead had, with OK Computer, peaked a little early. Unlike Roger Waters' similar plight with stadium-filling Pink Floyd, Thom didn't get all m

Masked:
By 1998 [MASKED] were at breaking point. Mentally and physically exhausted, [MASKED] – forever the sickly Victorian child – had been pushed too far by fame and adulation. Like many before him, the thrill of playing to a stadium full of unquestioningly adoring fans had become soured by the sheer numbing quality of constant public scrutiny. In a way, [MASKED] had, with OK Computer, peaked a little early. Unlike Roger Waters' similar plight with stadium-filling Pink Floyd, Thom 

In [45]:
# Save updated dataset with personnel and masked text columns.
# Original cleaned_text is preserved — masked version is in cleaned_text_masked.
out_path = DATA_DIR / "merged_dataset_masked.parquet"
df.to_parquet(out_path, index=False)
print(f"Saved: {out_path}")
print(f"Columns: {list(df.columns)}")


Saved: /Users/jamesemcnally/Documents/GitHub/critical-listener/datasets/merged_dataset_masked.parquet
Columns: ['dataset', 'review_id', 'entity_id', 'text', 'rating', 'album', 'artist', 'reviewer_name', 'reviewer_id', 'reviewer_type', 'reviewer_karma', 'published_on', 'source', 'source_url', 'cleaned_text', 'word_count', 'char_count', 'sentence_count', 'sentiment_score', 'sentiment_category', 'genre', 'ra_recommends', 'artist_norm', 'album_norm', 'input_with_prefix', 'input_no_prefix', 'personnel', 'cleaned_text_masked']
